# 🚀 Antigravity & Runpod Cloud Worker for Google Colab

This notebook connects your Google Colab GPU runtime (T4, L4, or A100) directly to your **Antigravity AI Agent** on your S25 Ultra / Termux workstation.

### Capabilities:
- ⚡ **Tri-Compute Coordination**: Antigravity coordinates between **Termux (Local S25 Ultra)**, **Google Colab (Training/Burst GPU)**, and **Runpod (Serverless/Pods)**.
- 🤗 **Hugging Face Hub Bridge**: Automatically push trained models, LoRAs, and weights from Colab directly to Hugging Face so Runpod and Termux can immediately pull them.
- 🔒 **Secure Remote Tunnel**: Exposes an execution API via Cloudflare / Ngrok for seamless Antigravity job dispatching.

In [ ]:
# Step 1: Verify GPU Acceleration
!nvidia-smi
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# Step 2: Install Worker Dependencies
!pip install -q fastapi uvicorn pyngrok huggingface_hub accelerate transformers torch pydantic

In [ ]:
# Step 3: Hugging Face Authentication
import os
from huggingface_hub import login

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = os.environ.get('HF_TOKEN')

if hf_token:
    login(token=hf_token)
    print("✅ Logged in to Hugging Face successfully!")
else:
    print("ℹ️ No HF_TOKEN found in Colab Secrets. Add 'HF_TOKEN' in Colab Secrets or enter it manually below:")
    # login()

In [ ]:
# Step 4: Antigravity Worker Server & Tunnel
import subprocess
import threading
import json
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import uvicorn

app = FastAPI(title="Antigravity Colab Worker")

class CommandRequest(BaseModel):
    command: str
    timeout: int = 300

class UploadRequest(BaseModel):
    local_path: str
    repo_id: str
    repo_type: str = "model"

@app.get("/status")
def get_status():
    import torch
    gpu_info = {}
    if torch.cuda.is_available():
        gpu_info = {
            "device": torch.cuda.get_device_name(0),
            "allocated_gb": torch.cuda.memory_allocated(0) / 1e9,
            "total_gb": torch.cuda.get_device_properties(0).total_memory / 1e9
        }
    return {
        "status": "ready",
        "cuda_available": torch.cuda.is_available(),
        "gpu": gpu_info
    }

@app.post("/exec")
def exec_command(req: CommandRequest):
    try:
        res = subprocess.run(req.command, shell=True, capture_output=True, text=True, timeout=req.timeout)
        return {
            "exit_code": res.returncode,
            "stdout": res.stdout,
            "stderr": res.stderr
        }
    except subprocess.TimeoutExpired:
        raise HTTPException(status_code=408, detail="Command timed out")
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/upload_to_hf")
def upload_to_hf(req: UploadRequest):
    from huggingface_hub import HfApi
    api = HfApi()
    if os.path.isdir(req.local_path):
        url = api.upload_folder(folder_path=req.local_path, repo_id=req.repo_id, repo_type=req.repo_type)
    else:
        url = api.upload_file(path_or_fileobj=req.local_path, path_in_repo=os.path.basename(req.local_path), repo_id=req.repo_id, repo_type=req.repo_type)
    return {"status": "uploaded", "url": url}

# Start background Uvicorn server
def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")

t = threading.Thread(target=run_server, daemon=True)
t.start()
print("✅ Worker server running locally on port 8000.")

In [ ]:
# Step 5: Expose Public Tunnel via Cloudflare Tunnel
# Downloads cloudflared and opens a quick tunnel to port 8000
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

import subprocess, time, re
proc = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"], stderr=subprocess.PIPE, text=True)

tunnel_url = None
for _ in range(30):
    line = proc.stderr.readline()
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        tunnel_url = match.group(0)
        break
    time.sleep(0.5)

if tunnel_url:
    print("=" * 60)
    print(f"🎉 COLAB WORKER CONNECTED!")
    print(f"Public Tunnel URL: {tunnel_url}")
    print("\n👉 In your Termux Antigravity shell, run:")
    print(f"ag-cloud colab connect {tunnel_url}")
    print("=" * 60)
else:
    print("Tunnel initialization taking longer. Check cloudflared output or try ngrok.")